In [0]:

CREATE OR REPLACE TABLE telecom_lakehouse.bronze.billing_invoices_raw
USING DELTA
COMMENT 'Synthetic raw telecom billing invoice data'
AS
SELECT
    CONCAT('INV', LPAD(CAST(i.id AS STRING), 9, '0')) AS invoice_id,

    s.subscription_id,
    s.customer_id,

    DATE_ADD(
        DATE'2025-01-01',
        CAST(PMOD(i.id * 7, 365) AS INT)
    ) AS invoice_date,

    DATE_ADD(
        DATE_ADD(
            DATE'2025-01-01',
            CAST(PMOD(i.id * 7, 365) AS INT)
        ),
        30
    ) AS due_date,

    CONCAT(
        '2025M',
        LPAD(
            CAST(
                MONTH(
                    DATE_ADD(
                        DATE'2025-01-01',
                        CAST(PMOD(i.id * 7, 365) AS INT)
                    )
                ) AS STRING
            ),
            2,
            '0'
        )
    ) AS billing_period,

    s.monthly_price AS base_amount,

    ROUND(PMOD(i.id * 17, 200) + 10, 2) AS additional_charges,

    ROUND(
        s.monthly_price + PMOD(i.id * 17, 200) + 10,
        2
    ) AS invoice_amount,

    CASE
        WHEN PMOD(i.id, 10) < 6 THEN 'PAID'
        WHEN PMOD(i.id, 10) < 8 THEN 'PARTIALLY_PAID'
        ELSE 'UNPAID'
    END AS invoice_status,

    CASE
        WHEN PMOD(i.id, 10) < 6
            THEN ROUND(
                s.monthly_price + PMOD(i.id * 17, 200) + 10,
                2
            )
        WHEN PMOD(i.id, 10) < 8
            THEN ROUND(
                (s.monthly_price + PMOD(i.id * 17, 200) + 10) * 0.5,
                2
            )
        ELSE 0
    END AS paid_amount,

    'BILLING_SYSTEM' AS source_system,
    'BATCH_001' AS ingest_batch_id,
    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM RANGE(1, 200001) i
INNER JOIN telecom_lakehouse.bronze.subscriptions_raw s
    ON s.subscription_id = CONCAT(
        'SUB',
        LPAD(
            CAST(PMOD(i.id * 37, 100000) + 1 AS STRING),
            8,
            '0'
        )
    );